In [2]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np

<img src="./multilayer_perceptron1.png" width="1500" align="left" />
<img src="./multilayer_perceptron2.png" width="1500" align="left" />
<img src="./multilayer_perceptron3.png" width="1500" align="left" />
<img src="./multilayer_perceptron4.png" width="1500" align="left" />

# 다층 퍼셉트론을 사용해서 XOR 문제를 해결한다.

## 피처와 레이블을 저장할 placeholder를 만든다.

XOR 연산은 입력 데이터를 X라고 할 경우 [[0, 0], [0, 1], [1, 0], [1, 1]]와 같은 경우의 수가 있고 총 4개의 입력 데이터가 각각 2개씩 피처를 가지고 있으므로 `X는 [4, 2]`의 형태로 정의할 수 있고, 입력값에 따른 레이블을 Y라고 할 경우 [[0], [1], [1], [0]]이므로 `Y는 [4, 1]`의 형태로 정의할 수 있다.

In [3]:
X = tf.placeholder(dtype=tf.float32, shape=[4, 2]) # 피처
Y = tf.placeholder(dtype=tf.float32, shape=[4, 1]) # 피처에 따른 레이블

## 다층 퍼셉트론의 1번째 히든 레이어를 만든다.

1번째 히든 레이어에서 2개의 입력(X)과 1개의 바이어스(B1)를 받아서 2개의 시그모이드 함수 출력(Z)을 2번째 히든 레이어로 전달한다.

1번째 히든 레이어에서 사용할 가중치와 바이어스를 랜덤하게 만든다.

In [6]:
# 2개의 입력을 받는 2개의 뉴런을 사용하기 위해 가중치를 [입력 데이터 개수, 뉴런 개수] 만큼으로 난수를 만든다.
W1 = tf.Variable(tf.random_uniform([2, 2], dtype=tf.float32))
# 각 뉴런은 1개의 바이어스를 가진다.
# zeros() 메소드는 모든 원소의 값이 0인 텐서(2차원 이상의 배열)를 만든다.
B1 = tf.Variable(tf.zeros([2]))
# 활성화 함수(시그모이드 함수)를 이용해서 1번째 히든 레이어의 출력을 리턴한다.
Z = tf.sigmoid(tf.matmul(X, W1) + B1)

In [41]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())
print(sess.run(W1))
print(sess.run(B1))
x_train = [[0, 0], [0, 1], [1, 0], [1, 1]]
print(sess.run(Z, feed_dict={X: x_train}))

[[0.20247781 0.55869496]
 [0.18648016 0.64603674]]
[0. 0.]
[[0.5        0.5       ]
 [0.5464854  0.65611684]
 [0.5504472  0.63615054]
 [0.5960318  0.76936543]]


## 다층 퍼셉트론의 2번째 히든 레이어를 만든다.

2번째 히든 레이어에서 1번째 히든 레이어의 출력값(Z)을 입력으로 받고 1개의 바이어스(B2)를 받아서 1개의 시그모이드 함수 결과(Y_hat)를 출력한다.

2번째 히든 레이어에서 사용할 가중치와 바이어스를 랜덤하게 만든다.

In [42]:
# 2개의 입력을 받는 1개의 뉴런을 사용하기 위해 가중치를 [입력 데이터 개수, 뉴런 개수] 만큼으로 난수를 만든다.
W2 = tf.Variable(tf.random_uniform([2, 1], dtype=tf.float32))
# 각 뉴런은 1개의 바이어스를 가진다.
B2 = tf.Variable(tf.zeros([1]))
# 활성화 함수(시그모이드 함수)를 이용해서 2번째 히든 레이어의 출력을 리턴한다.
Y_hat = tf.sigmoid(tf.matmul(Z, W2) + B2)

In [43]:
sess.run(tf.global_variables_initializer())
print(sess.run(W2))
print(sess.run(B2))
print(sess.run(Y_hat, feed_dict={X: x_train}))

[[0.35834265]
 [0.24842358]]
[0.]
[[0.57526934]
 [0.6038902 ]
 [0.5837866 ]
 [0.6106808 ]]


## 역전파 시키기 위해서 손실 함수(크로스 엔트로피)를 만들고 경사 하강법으로 모델의 가중치와 바이어스를 최적화 한다.

이진 크로스 엔트로피

크로스 엔트로피 오차 함수는 주로 분류 문제에서 모델이 예측한 확률 분포와 실제 정답 분포 간의 차를 측정하는 손실 함수로 딥러닝에서 로지스틱 회귀나 다중 분류 클래스 분류 모델을 학습시킬 때 가장 널리 사용된다.

In [44]:
loss = tf.reduce_mean(
    -1 * (Y * tf.log(Y_hat) + (1.0 - Y) * tf.log(1.0 - Y_hat))
)

경사 하강법

In [45]:
train = tf.train.GradientDescentOptimizer(0.05).minimize(loss)

## 다층 퍼셉트론을 학습하고 테스트한다.

In [48]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

x_train = [[0, 0], [0, 1], [1, 0], [1, 1]] # 피처
y_train = [[0], [1], [1], [0]] # 피처에 따른 레이블

for epoch in range(20001):
    _, _Y_hat = sess.run([train, Y_hat], feed_dict={X: x_train, Y: y_train})
    if epoch % 5000 == 0:
        print(f'epoch: {epoch}')
        print(f'output\n{_Y_hat}')
        print('-' * 100)

epoch: 0
output
[[0.5610544 ]
 [0.5776214 ]
 [0.56917167]
 [0.5840062 ]]
----------------------------------------------------------------------------------------------------
epoch: 5000
output
[[0.34468433]
 [0.5521611 ]
 [0.55546427]
 [0.5831499 ]]
----------------------------------------------------------------------------------------------------
epoch: 10000
output
[[0.09445039]
 [0.8874788 ]
 [0.88733804]
 [0.15230967]]
----------------------------------------------------------------------------------------------------
epoch: 15000
output
[[0.03538821]
 [0.96984345]
 [0.9698122 ]
 [0.03501211]]
----------------------------------------------------------------------------------------------------
epoch: 20000
output
[[0.02099319]
 [0.98343015]
 [0.983417  ]
 [0.01841171]]
----------------------------------------------------------------------------------------------------
